# Lab Work - 6.7

## Q1: Code It

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from sklearn import tree
import numpy as np

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))

In [ ]:
# Fit Decision Tree
clf = DecisionTreeClassifier(criterion='gini', random_state=42)
clf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Confusion Matrix and Report
print("Confusion Matrix:")
print(confusion_matrix(y_test, clf.predict(X_test)))
print("\nClassification Report:")
print(classification_report(y_test, clf.predict(X_test), target_names=target_names))

In [ ]:
# Tree Structure
print(tree.export_text(clf, feature_names=feature_names))

In [ ]:
# Feature Importances
importances = clf.feature_importances_
for name, imp in zip(feature_names, importances):
    print(f"{name}: {imp:.4f}")
print("Root split feature:", feature_names[np.argmax(importances)])

In [ ]:
# AUC-ROC (one-vs-rest)
y_test_bin = label_binarize(y_test, classes=[0,1,2])
y_pred_prob = clf.predict_proba(X_test)

auc_scores = []
for i in range(3):
    auc = roc_auc_score(y_test_bin[:, i], y_pred_prob[:, i])
    auc_scores.append(auc)
    print(f"Class {target_names[i]} AUC: {auc:.4f}")

print("Lowest AUC class:", target_names[np.argmin(auc_scores)])

## Q2: Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [1,2,3,4,5,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1_macro')
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)
print("Number of combinations evaluated:", len(grid_search.cv_results_['params']))

In [ ]:
# Fit best estimator
best_clf = grid_search.best_estimator_
best_clf.fit(X_train, y_train)
print("Test Accuracy:", accuracy_score(y_test, best_clf.predict(X_test)))
print("F1-macro:", classification_report(y_test, best_clf.predict(X_test), output_dict=True)['macro avg']['f1-score'])

In [ ]:
# Max depth sweep (visualise overfitting/underfitting)
depths = range(1,11)
train_f1s = []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_f1s.append(classification_report(y_train, dt.predict(X_train), output_dict=True)['macro avg']['f1-score'])

plt.plot(depths, train_f1s, label='Train F1-macro', marker='o')
plt.xlabel('max_depth')
plt.ylabel('F1-macro')
plt.title('Effect of max_depth on Training Performance')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Complexity metrics
for md in [2,5,None]:
    dt = DecisionTreeClassifier(max_depth=md, random_state=42)
    dt.fit(X_train, y_train)
    print(f"max_depth={md}: leaves={dt.get_n_leaves()}, depth={dt.get_depth()}")

## Q3: Visualise

In [ ]:
# Visualise tuned tree
plt.figure(figsize=(15,10))
tree.plot_tree(best_clf, feature_names=feature_names, class_names=target_names, filled=True, rounded=True)
plt.title('Decision Tree (Tuned)')
plt.show()

In [ ]:
# Decision boundary (petal length & width)
X2 = X[:, [2, 3]]
X_train2, _, y_train2, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)

clf2 = DecisionTreeClassifier(max_depth=3, random_state=42)
clf2.fit(X_train2, y_train2)

x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.4)
plt.scatter(X_train2[:,0], X_train2[:,1], c=y_train2, edgecolor='k', cmap=plt.cm.Set1)
plt.xlabel('Petal Length')
plt.ylabel('Petal Width')
plt.title('Decision Boundary (max_depth=3)')
plt.show()

In [ ]:
# Feature Importances Bar Chart
importances = best_clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), np.array(feature_names)[indices], rotation=45)
plt.title('Feature Importances (Decision Tree)')
plt.show()

## Q4: Deep Intuition

1. **Overfitting Diagnosis & Fixes**  
   Overfitting in a decision tree is diagnosed when the model performs much better on training data than on test data, and the tree becomes very deep with many leaves. Common fixes include pruning the tree, limiting complexity with `max_depth`, `min_samples_split`, or `min_samples_leaf`, and using cross-validation to choose hyperparameters. These techniques reduce variance and improve generalisation by preventing the tree from memorising noise in the training set.

2. **Gini vs Entropy**  
   Gini impurity and entropy are both measures of node impurity used to choose splits. Gini impurity is simpler and tends to prefer larger pure nodes, while entropy is based on information theory and can be slightly more sensitive to class probability differences. In practice they usually produce similar trees, but Gini is often faster to compute and is the default criterion in scikit-learn.

3. **Decision Tree vs Logistic Regression**  
   A decision tree can model nonlinear decision boundaries and handle categorical features without feature scaling, while logistic regression is a linear model that is best when classes are linearly separable. Logistic regression tends to generalise better on simple problems and is more stable for small datasets, whereas decision trees can capture complex interactions but can overfit unless regularised.